In [3]:
import torch
import datasets
from datasets import load_dataset
dts = load_dataset("dair-ai/emotion")

In [4]:
import sklearn
print(sklearn.__version__)

1.9.1


In [5]:
train_dts = dts["train"]
val_dts = dts["validation"]
test_dts = dts["test"]

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(stop_words = "english")

In [7]:
featurised = vectorizer.fit_transform(train_dts['text'])
featurised_test = vectorizer.transform(test_dts['text']) 

In [8]:
from sklearn.linear_model import LogisticRegression
mlr = LogisticRegression()
mlr.fit(featurised , train_dts['label'])

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

In [9]:
from sklearn import metrics
predictions = mlr.predict(featurised_test)

In [10]:
print("--- Classification Report ---")
print(metrics.classification_report(test_dts['label'], predictions))

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.90      0.92      0.91       581
           1       0.85      0.95      0.90       695
           2       0.78      0.64      0.71       159
           3       0.89      0.81      0.85       275
           4       0.87      0.82      0.84       224
           5       0.92      0.50      0.65        66

    accuracy                           0.87      2000
   macro avg       0.87      0.77      0.81      2000
weighted avg       0.87      0.87      0.86      2000



In [11]:
import transformers 
from transformers import AutoModelForSequenceClassification , AutoConfig , AutoTokenizer , DataCollatorWithPadding , set_seed 
set_seed(42)
model_name = "roberta-base"
model = AutoModelForSequenceClassification.from_pretrained(model_name , num_labels = 6 )
tokenizer = AutoTokenizer.from_pretrained(model_name)
configs = AutoConfig.from_pretrained(model_name)
print(configs)
def fn(batch_dict):
    text_list = list(batch_dict['text'])
    tokens = tokenizer(text_list , truncation = True , max_length = 128)
    return tokens

W1031 16:58:06.542000 2452 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaConfig {
  "add_cross_attention": false,
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "tie_word_embeddings": true,
  "transformers_version": "5.21.0",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}



In [12]:
dts_1 = dts.map(fn, batched = True)

In [13]:
collator = DataCollatorWithPadding(tokenizer)

In [14]:
%pip install "accelerate>=1.1.0"

Note: you may need to restart the kernel to use updated packages.


In [15]:
from transformers import TrainingArguments , Trainer

In [16]:
training_args = TrainingArguments(
    learning_rate = 2e-5,
    output_dir="./results",               # Checkpoint storage directory
    num_train_epochs=10,                   # Total epochs
    per_device_train_batch_size=16,       # Batch size per device during training
    per_device_eval_batch_size=64,        # Batch size per device during evaluation
    warmup_steps=500,                     # Number of warmup steps for learning rate scheduler
    weight_decay=0.01,                    # Strength of weight decay
    logging_steps=10,                     # Log every X updates steps
    eval_strategy="epoch",                # Evaluate at the end of each epoch
    save_strategy="epoch",                # Save a checkpoint at the end of each epoch
    load_best_model_at_end=True,          # Load the best model when finished training
    fp16=True,
    seed = 42,
    metric_for_best_model = "macro_f1"
)

In [17]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    labels = p.label_ids
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

In [18]:
trainer = Trainer(
    model=model,                          # The PreTrainedModel to train
    args=training_args,                   # The defined training arguments
    train_dataset=dts_1['train'],          # Training dataset
    eval_dataset=dts_1['validation'],      # Evaluation dataset
    compute_metrics=compute_metrics,        # Function to calculate metrics (e.g., accuracy)
    data_collator = collator,
    processing_class = tokenizer
)

In [19]:
import time, torch
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
trainer.train()
print(f"{time.time() - t0:.0f} s")
print(f"{torch.cuda.max_memory_allocated() / 1e9:.2f} GB peak VRAM")

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.352873,0.275051,0.912000,0.886438
2,0.273406,0.214146,0.935000,0.901639
3,0.159465,0.173953,0.937500,0.913342
4,0.179495,0.155840,0.939500,0.914601
5,0.082287,0.207890,0.938000,0.913189
6,0.085988,0.210164,0.937000,0.912628
7,0.099060,0.253092,0.933000,0.902180
8,0.044960,0.311297,0.937000,0.911580
9,0.107079,0.333747,0.934500,0.905638
10,0.028438,0.346070,0.935000,0.908766


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (2 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

605 s
2.33 GB peak VRAM


In [20]:
trainer.save_model("my_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [22]:
print(torch.cuda.memory_allocated() / 1e9, "GB in use by tensors")

1.517885952 GB in use by tensors


In [23]:
import os
print(os.listdir("my_model"))

['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']


In [1]:
import torch
from transformers import pipeline

clf = pipeline(
    "text-classification",
    model="my_model",
    device=0 if torch.cuda.is_available() else -1,
)

print(clf("This movie was great"))

W1031 17:16:46.597000 32000 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': 'LABEL_1', 'score': 0.9861329793930054}]
